# Test GNSS Pseudorange errors with the Loop Paramters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt

In [ ]:
Bp = 1.0  # Carrier loop noise bandwidth [Hz]  (higher Bp (5.0 Hz) for closer distance to Earth)
T = 20e-3  # Tracking loop integration time [s]
b = 2.0  # normalized bandwidth [Hz]
Bn = 0.7  # Code loop noise bandwidth [Hz]
Bf = 0.2  # Frequency loop noise bandwidth [Hz]
D = 0.1  # Early-to-late correlator spacing (chips, correlator spacing should be larger than 0.25 chips)

# Tara's paper settings
# Bp = 1.0  # Carrier loop noise bandwidth [Hz]  (higher Bp (5.0 Hz) for closer distance to Earth)
# T = 10e-3  # Tracking loop integration time [s]
# b = 1.0  # normalized bandwidth [Hz]
# Bn = 2.0   # Code loop noise bandwidth [Hz]
# Bf = 0.2  # Frequency loop noise bandwidth [Hz]
# D = 0.1  # Early-to-late correlator spacing (chips, correlator spacing should be larger than 0.25 chips)

In [ ]:
gnss_freq_map = {
    "L1": 1575.42e6,
    "L2": 1227.60e6,
    "L5": 1176.45e6,
    "E1": 1575.42e6,
    "E6": 1278.75e6,
    "E5": 1191.795e6,
    "E5a": 1176.45e6,
    "E5b": 1207.14e6,
}

gnss_rc_map = {
    "L1": 1.023e6,
    "L2": 0.5115e6,
    "L5": 10.23e6,
    "E1": 1.023e6,
    "E6": 0.5115e6,
    "E5": 10.23e6,
    "E5a": 10.23e6,
    "E5b": 10.23e6,
}

In [ ]:
def compute_gnss_pseudorange_noise(CN0_dB, freq):
    CN0 = 10 ** (CN0_dB / 10)
    Rc = gnss_rc_map[freq]  # Chip rate in Hz
    Bfe = b * Rc
    Tc = 1.0 / Rc
    C = pnt.C  # Speed of light in m/s

    sigma = np.zeros_like(CN0)

    # Case 1: Wide spacing discriminator
    if D >= (np.pi * Rc / Bfe):
        print("Wide spacing discriminator")
        sigma = np.sqrt(Bn / (2.0 * CN0) * D * (1.0 + 2.0 / (T * CN0 * (2 - D))))
    elif D > (Rc / Bfe):
        print("Narrow spacing discriminator")
        tmp1 = Bn / (2.0 * CN0)
        tmp2 = 1.0 / (Bfe * Tc) + Bfe * Tc / (np.pi - 1) * (D - 1.0 / (Bfe * Tc)) ** 2
        tmp3 = 1.0 + 2.0 / (T * CN0 * (2 - D))
        sigma = np.sqrt(tmp1 * tmp2 * tmp3)
    else:
        print("Very narrow spacing discriminator")
        sigma = np.sqrt(Bn / (2.0 * CN0) * (1.0 / (Bfe * Tc)) * (1.0 + 1.0 / (T * CN0)))

    return sigma * (C * Tc)


def compute_gnss_pseudorangerate_noise(CN0_dB, freq):
    # https://theses.eurasip.org/media/theses/documents/padma-bolla-advanced-tracking-loop-architectures-for-multi-frequency-gnss-receiver.pdf?utm_source=chatgpt.com
    f = gnss_freq_map[freq]
    lambda_ = pnt.C / f  # Wavelength of the GNSS signal

    CN0 = 10 ** (CN0_dB / 10)
    F = 2

    return lambda_ / (2 * np.pi * T) * np.sqrt(4 * Bf / CN0 * (1 + 1.0 / (T * CN0)))


def compute_gnss_carrier_phase_noise(CN0_dB, freq):
    CN0 = 10 ** (CN0_dB / 10)
    f = gnss_freq_map[freq]
    lambda_ = pnt.C / f  # Wavelength of the GNSS signal

    return lambda_ / (2 * np.pi) * np.sqrt(Bp / CN0 * (1.0 + 1.0 / (2 * T * CN0)))

comapre with lugre receiver profile
https://ntrs.nasa.gov/api/citations/20220010106/downloads/AAS_2022_LuGRE_Analysis_STRIVES_v2.pdf

In [ ]:
CN0_dB = np.linspace(15, 50, 100)

signals = ["L1", "L5"]

fig, ax = plt.subplots(1, 2, figsize=(8, 3))

for sig in signals:
    dll_noise = compute_gnss_pseudorange_noise(CN0_dB, sig)
    pll_noise = compute_gnss_pseudorangerate_noise(CN0_dB, sig)
    carr_phase_noise = compute_gnss_carrier_phase_noise(CN0_dB, sig)

    print(f"GNSS Pseudorange Noise ({sig}) [m]:", dll_noise)
    ax[0].plot(CN0_dB, dll_noise, label=sig)
    ax[0].set_title(f"GNSS Pseudorange Noise vs CN0")
    ax[0].set_xlabel("CN0 (dB-Hz)")
    ax[0].set_ylabel("Pseudorange Noise (m)")
    ax[0].set_xlim(15, 50)
    ax[0].set_ylim(0, 30)
    ax[0].grid(True)
    # set a minor dashed y grid of 1
    ax[0].yaxis.set_minor_locator(plt.MultipleLocator(1))
    ax[0].grid(which="minor", linestyle="--", alpha=0.5)
    # set a minor dashed x grid of 2
    ax[0].xaxis.set_minor_locator(plt.MultipleLocator(2))
    ax[0].grid(which="minor", linestyle="--", alpha=0.5)
    ax[0].legend()

    # ax[1].plot(CN0_dB, pll_noise)
    # ax[1].set_title(f"GNSS Pseudorangerate Noise vs CN0 ({sig})")
    # ax[1].set_xlabel("CN0 (dB-Hz)")
    # ax[1].set_ylabel("Pseudorangerate Noise (m/s)")
    # # set a minor dashed y grid of 0.1
    # ax[1].yaxis.set_minor_locator(plt.MultipleLocator(0.1))
    # ax[1].grid(which="minor", linestyle="--", alpha=0.5)
    # # set a minor dashed x grid of 2
    # ax[1].xaxis.set_minor_locator(plt.MultipleLocator(2))
    # ax[1].grid(which="minor", linestyle="--", alpha=0.5)
    # ax[1].grid()

    ax[1].plot(CN0_dB, carr_phase_noise * 1000, label=sig)  # convert to mm/s
    ax[1].set_title(f"GNSS Carrier Phase Noise vs CN0 ({sig})")
    ax[1].set_xlabel("CN0 (dB-Hz)")
    ax[1].set_ylabel("Carrier Phase Noise (mm)")
    ax[1].set_xlim(15, 50)
    ax[1].grid(True)
    # set a minor dashed y grid of 1
    ax[1].yaxis.set_minor_locator(plt.MultipleLocator(1))
    ax[1].grid(which="minor", linestyle="--", alpha=0.5)
    # set a minor dashed x grid of 2
    ax[1].xaxis.set_minor_locator(plt.MultipleLocator(2))
    ax[1].grid(which="minor", linestyle="--", alpha=0.5)
    ax[1].legend()

    plt.tight_layout()

figname = (
    pnt.get_output_dir() / "iono_delay" / "gnss_plots" / "gnss_tracking_errors.pdf"
)
plt.savefig(figname)
plt.show()

In [ ]:
0.02 * 0.1903 * 1 * 1000  # in mm